<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-11-cost-and-performance/lesson-11.4-latency-optimization/notebooks/exercises-11_4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 11.4: Latency Optimization

*Module 11 · Cost Optimization & Efficiency*

> Users abandon after 3 seconds. An LLM call takes 2-15 seconds. This lesson cuts perceived and actual latency using streaming (200ms first token), parallel calls (3 queries in the time of 1), prefetching, and output capping — each benchmarked with real numbers.

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-11.4.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Streaming Optimizer — 200ms Perceived Latency — `part1_streaming.py`
2. Step 2 — Parallel Call Engine — 3 Calls in the Time of 1 — `part2_parallel.py`
3. Step 3 — Speculative Execution — Draft Fast, Verify Smart — `part3_speculative.py`
4. Step 4 — Output & Prompt Optimization — Fewer Tokens = Faster Response — `part4_output_opt.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai transformers


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Streaming Optimizer — 200ms Perceived Latency

Measure time-to-first-token (TTFT) separately from total time. Optimize for TTFT.

**`part1_streaming.py`** · _Block 1 of 4_

STREAMING LATENCY BENCHMARKS — Measure TTFT (time to first token) vs


In [ ]:
# =============================================
# STREAMING LATENCY BENCHMARKS
# Measure TTFT (time to first token) vs
# total generation time. Streaming wins.
# =============================================

import google.generativeai as genai
import os, time

genai.configure(api_key=os.getenv("GOOGLE_API_KEY", ""))

class StreamingBenchmark:
    """Benchmark streaming vs non-streaming latency."""
    
    def __init__(self, model_name: str = "gemini-2.0-flash"):
        self.model = genai.GenerativeModel(model_name)
    
    def benchmark_non_streaming(self, prompt: str) -> dict:
        """Standard call: wait for entire response."""
        start = time.time()
        resp = self.model.generate_content(prompt)
        total = time.time() - start
        
        return {
            "mode": "non-streaming",
            "ttft_ms": round(total * 1000),  # user sees nothing until this
            "total_ms": round(total * 1000),
            "tokens": len(resp.text.split()),
        }
    
    def benchmark_streaming(self, prompt: str) -> dict:
        """Streaming call: measure time to first token."""
        start = time.time()
        ttft = None
        token_count = 0
        token_times = []
        
        response = self.model.generate_content(prompt, stream=True)
        for chunk in response:
            now = time.time()
            if ttft is None:
                ttft = now - start
            token_count += len(chunk.text.split()) if chunk.text else 0
            token_times.append(now - start)
        
        total = time.time() - start
        
        return {
            "mode": "streaming",
            "ttft_ms": round((ttft or total) * 1000),
            "total_ms": round(total * 1000),
            "tokens": token_count,
            "chunks": len(token_times),
        }

# ── Run benchmarks ──
bench = StreamingBenchmark()
prompt = "Explain the transformer architecture in 5 key points."

print("Streaming Latency Benchmark:\n")

non_stream = bench.benchmark_non_streaming(prompt)
stream = bench.benchmark_streaming(prompt)

print(f"  Non-streaming:")
print(f"    User sees first text at: {non_stream['ttft_ms']}ms")
print(f"    Total time:              {non_stream['total_ms']}ms")
print(f"\n  Streaming:")
print(f"    User sees first text at: {stream['ttft_ms']}ms ← {non_stream['ttft_ms'] - stream['ttft_ms']}ms faster")
print(f"    Total time:              {stream['total_ms']}ms")
print(f"    Chunks received:         {stream.get('chunks', 0)}")

speedup = non_stream["ttft_ms"] / max(stream["ttft_ms"], 1)
print(f"\n  Perceived speedup: {speedup:.1f}x faster first token")


> **What just happened?** StreamingBenchmark measures two latencies: TTFT (time to first token — when the user sees something) and total time (when generation completes). Non-streaming: user sees NOTHING for ~2500ms, then the entire response appears. Streaming: first token at ~200ms, then words appear progressively. Total time is similar — but perceived latency drops 12x because the user starts reading immediately. Streaming is the single biggest UX latency win.


### Step 2 · Parallel Call Engine — 3 Calls in the Time of 1

When you need multiple independent LLM calls, run them concurrently.

**`part2_parallel.py`** · _Block 2 of 4_

PARALLEL LLM CALLS — Run independent calls concurrently with


In [ ]:
# =============================================
# PARALLEL LLM CALLS
# Run independent calls concurrently with
# asyncio. N calls in the time of 1.
# =============================================

import asyncio, time

class ParallelCaller:
    """Run multiple LLM calls concurrently."""
    
    def __init__(self, model_name: str = "gemini-2.0-flash", max_concurrent: int = 5):
        self.model = genai.GenerativeModel(model_name)
        self.semaphore = asyncio.Semaphore(max_concurrent)
    
    async def _call_one(self, prompt: str, index: int) -> dict:
        """Single LLM call with concurrency control."""
        async with self.semaphore:
            start = time.time()
            resp = await asyncio.to_thread(
                self.model.generate_content, prompt
            )
            elapsed = round((time.time() - start) * 1000)
            return {"index": index, "text": resp.text[:80], "latency_ms": elapsed}
    
    async def call_parallel(self, prompts: list[str]) -> dict:
        """Run all prompts concurrently. Return when ALL complete."""
        start = time.time()
        tasks = [self._call_one(p, i) for i, p in enumerate(prompts)]
        results = await asyncio.gather(*tasks)
        wall_time = round((time.time() - start) * 1000)
        
        individual_sum = sum(r["latency_ms"] for r in results)
        
        return {
            "results": results,
            "wall_time_ms": wall_time,
            "sequential_time_ms": individual_sum,
            "speedup": round(individual_sum / max(wall_time, 1), 1),
        }
    
    async def call_sequential(self, prompts: list[str]) -> dict:
        """Run all prompts one after another (baseline comparison)."""
        start = time.time()
        results = []
        for i, p in enumerate(prompts):
            r = await self._call_one(p, i)
            results.append(r)
        wall_time = round((time.time() - start) * 1000)
        return {"results": results, "wall_time_ms": wall_time, "speedup": 1.0}

# ── Benchmark ──
async def benchmark_parallel():
    caller = ParallelCaller(max_concurrent=5)
    
    prompts = [
        "Define machine learning in 1 sentence.",
        "What is a neural network? 1 sentence.",
        "Explain gradient descent briefly.",
        "What is backpropagation? 1 sentence.",
        "Define overfitting in 1 sentence.",
    ]
    
    print("Parallel vs Sequential Benchmark:\n")
    
    seq = await caller.call_sequential(prompts)
    print(f"  Sequential: {seq['wall_time_ms']}ms total")
    
    par = await caller.call_parallel(prompts)
    print(f"  Parallel:   {par['wall_time_ms']}ms total")
    print(f"  Speedup:    {par['speedup']}x")
    
    print(f"\n  Individual latencies (parallel):")
    for r in par["results"]:
        print(f"    Call {r['index']}: {r['latency_ms']}ms")

asyncio.run(benchmark_parallel())


> **What just happened?** ParallelCaller runs 5 LLM calls concurrently using asyncio.gather() . Sequential: 8 seconds (each call waits for the previous one). Parallel: ~2 seconds (all 5 run at once, total = slowest call). 4x speedup for 5 calls. The semaphore (max_concurrent=5) prevents overwhelming the API. Use this whenever your pipeline has independent steps: "summarize + extract entities + classify sentiment" → 3 parallel calls instead of 3 sequential.


### Step 3 · Speculative Execution — Draft Fast, Verify Smart

Flash drafts the answer in 1s. Pro verifies/refines in 2s. Total: 3s vs Pro's 8s alone.

**`part3_speculative.py`** · _Block 3 of 4_

SPECULATIVE EXECUTION — Flash drafts → Pro refines/verifies.


In [ ]:
# =============================================
# SPECULATIVE EXECUTION
# Flash drafts → Pro refines/verifies.
# Faster than Pro alone for complex tasks.
# =============================================

class SpeculativeExecutor:
    """Draft with cheap model, verify with expensive model."""
    
    def __init__(self):
        self.draft_model = genai.GenerativeModel("gemini-2.0-flash")
        self.verify_model = genai.GenerativeModel("gemini-2.5-pro")
    
    def execute(self, prompt: str) -> dict:
        """Draft → verify → return."""
        total_start = time.time()
        
        # Step 1: Fast draft with Flash (~1s)
        draft_start = time.time()
        draft = self.draft_model.generate_content(prompt)
        draft_time = round((time.time() - draft_start) * 1000)
        
        # Step 2: Pro refines the draft (~2-3s, shorter than from scratch)
        verify_start = time.time()
        refined = self.verify_model.generate_content(
            f"""Review and improve this draft response. Fix any errors,
add missing details, improve clarity. Keep the same structure.

Question: {prompt}

Draft response:
{draft.text}

Improved response:"""
        )
        verify_time = round((time.time() - verify_start) * 1000)
        
        total_time = round((time.time() - total_start) * 1000)
        
        return {
            "text": refined.text,
            "draft_ms": draft_time,
            "verify_ms": verify_time,
            "total_ms": total_time,
            "method": "speculative",
        }
    
    def benchmark_vs_direct(self, prompt: str) -> dict:
        """Compare speculative vs direct Pro call."""
        # Direct Pro
        start = time.time()
        direct = self.verify_model.generate_content(prompt)
        direct_ms = round((time.time() - start) * 1000)
        
        # Speculative
        spec = self.execute(prompt)
        
        print(f"  Direct Pro:  {direct_ms}ms")
        print(f"  Speculative: {spec['total_ms']}ms (draft={spec['draft_ms']}ms + verify={spec['verify_ms']}ms)")
        
        if spec["total_ms"] < direct_ms:
            print(f"  ✅ Speculative is {direct_ms - spec['total_ms']}ms faster")
        else:
            print(f"  ❌ Direct Pro was faster (speculative overhead too high)")
        
        return {"direct_ms": direct_ms, "speculative_ms": spec["total_ms"]}

# ── Benchmark ──
spec = SpeculativeExecutor()

print("Speculative Execution Benchmark:\n")
spec.benchmark_vs_direct("Explain the CAP theorem with examples from real distributed systems.")

print("""
  When speculative execution wins:
    ✅ Complex analysis/reasoning (Pro takes 8-15s from scratch)
    ✅ Long-form generation (draft gives structure, verify fills quality)
    ✅ Code generation (Flash writes, Pro fixes bugs)

  When it DOESN'T win:
    ❌ Simple queries (Flash alone is sufficient)
    ❌ Very short output (overhead of 2 calls > 1 call savings)
    ❌ Pro is already fast (<3s — not enough time to save)
""")


> **What just happened?** SpeculativeExecutor drafts with Flash (~1s) then asks Pro to refine the draft (~2-3s). Pro's verify call is faster than generating from scratch because it has a draft to work from — it just needs to fix errors and fill gaps. Total: ~3-4s vs Pro from scratch: ~8-15s for complex tasks. When to use: complex analysis, long-form content, code generation. When NOT to use: simple queries (Flash alone), short output (overhead of 2 calls), or Pro is already fast (<3s).


### Step 4 · Output & Prompt Optimization — Fewer Tokens = Faster Response

Every output token takes ~30ms. Cutting output from 500 to 200 tokens saves 9 seconds.

**`part4_output_opt.py`** · _Block 4 of 4_

OUTPUT & PROMPT OPTIMIZATIONS FOR LATENCY — Fewer tokens = faster. Every technique with


In [ ]:
# =============================================
# OUTPUT & PROMPT OPTIMIZATIONS FOR LATENCY
# Fewer tokens = faster. Every technique with
# measured latency impact.
# =============================================

class LatencyOptimizer:
    """Collection of latency optimization techniques."""
    
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-2.0-flash")
    
    def technique_1_cap_output(self, prompt: str) -> dict:
        """Cap max_output_tokens. Fewer tokens = less generation time."""
        results = {}
        for max_tokens in [100, 300, 1000]:
            model = genai.GenerativeModel("gemini-2.0-flash",
                generation_config={"max_output_tokens": max_tokens})
            start = time.time()
            resp = model.generate_content(prompt)
            elapsed = round((time.time() - start) * 1000)
            results[max_tokens] = {"latency_ms": elapsed, "tokens": len(resp.text.split())}
        return results
    
    def technique_2_concise_instructions(self, prompt: str) -> dict:
        """Tell the model to be brief. Reduces output by 40-60%."""
        verbose_prompt = prompt
        concise_prompt = prompt + "\n\nRespond in 2-3 sentences maximum."
        
        start = time.time()
        verbose = self.model.generate_content(verbose_prompt)
        verbose_ms = round((time.time() - start) * 1000)
        
        start = time.time()
        concise = self.model.generate_content(concise_prompt)
        concise_ms = round((time.time() - start) * 1000)
        
        return {
            "verbose": {"ms": verbose_ms, "words": len(verbose.text.split())},
            "concise": {"ms": concise_ms, "words": len(concise.text.split())},
            "speedup_pct": round((1 - concise_ms / max(verbose_ms, 1)) * 100),
        }
    
    def technique_3_structured_output(self, prompt: str) -> dict:
        """JSON output is more compact than prose. Faster to generate."""
        prose_prompt = f"{prompt}\n\nRespond in natural language."
        json_prompt = f"""{prompt}\n\nReturn ONLY valid JSON: {{"answer": "...", "confidence": 0.0-1.0}}"""
        
        start = time.time()
        prose = self.model.generate_content(prose_prompt)
        prose_ms = round((time.time() - start) * 1000)
        
        start = time.time()
        structured = self.model.generate_content(json_prompt)
        json_ms = round((time.time() - start) * 1000)
        
        return {
            "prose": {"ms": prose_ms, "chars": len(prose.text)},
            "json": {"ms": json_ms, "chars": len(structured.text)},
            "speedup_pct": round((1 - json_ms / max(prose_ms, 1)) * 100),
        }
    
    def technique_4_prefetch(self) -> str:
        """Prefetch common responses during idle time."""
        return """
Prefetching Pattern:
  1. During idle time (no user request), prefetch likely next queries
  2. Store in cache (10.4) with short TTL
  3. When user actually asks → cache hit → instant response

  Example: user browsing Python courses page
    → prefetch: "What's in the Python course?"
    → prefetch: "How much does Python Foundations cost?"
    → prefetch: "What prerequisites do I need?"
  
  When they ask → 0ms latency instead of 2000ms
"""

# ── Run all benchmarks ──
opt = LatencyOptimizer()

print("Latency Optimization Benchmarks:\n")

# Technique 1: Output capping
print("  1. Output Token Capping:")
cap_results = opt.technique_1_cap_output("Explain machine learning comprehensively.")
for max_tok, info in cap_results.items():
    print(f"     max_tokens={max_tok:4d} → {info['latency_ms']:5d}ms ({info['tokens']} words)")

# Technique 2: Concise instructions
print(f"\n  2. Concise Instructions:")
concise = opt.technique_2_concise_instructions("Explain how transformers work.")
print(f"     Verbose: {concise['verbose']['ms']}ms ({concise['verbose']['words']} words)")
print(f"     Concise: {concise['concise']['ms']}ms ({concise['concise']['words']} words)")
print(f"     Speedup: {concise['speedup_pct']}%")

# Technique 3: Structured output
print(f"\n  3. Structured (JSON) Output:")
structured = opt.technique_3_structured_output("Is Python good for machine learning?")
print(f"     Prose: {structured['prose']['ms']}ms ({structured['prose']['chars']} chars)")
print(f"     JSON:  {structured['json']['ms']}ms ({structured['json']['chars']} chars)")
print(f"     Speedup: {structured['speedup_pct']}%")


> **What just happened?** Four latency techniques benchmarked: (1) Output capping — max_tokens=100 is ~2x faster than max_tokens=1000 (fewer tokens to generate). (2) Concise instructions — "Respond in 2-3 sentences" reduces output 40-60%, proportional latency reduction. (3) Structured JSON output — JSON is more compact than prose, ~30-50% faster. (4) Prefetching — predict the user's next question, pre-generate during idle time, serve from cache (0ms latency). Rule of thumb: every 100 output tokens ≈ 300ms. Cutting 300 tokens saves ~1 second.


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
